IQAE validating the T transition epsilon, noisless condition


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json

from qiskit.circuit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem
from qiskit.primitives import Sampler


print("All imports OK ✅")

p_true = 0.1 ########################################################################################################

class BernoulliA(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)

class BernoulliQ(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)

    def power(self, k):
        q_k = QuantumCircuit(1)
        # ---------------------------------------------------------
        # REWRITTEN: Apply base rotation k times separated by barriers
        # to ensure the noise model acts k times.
        # ---------------------------------------------------------
        for _ in range(k):
            q_k.ry(2 * self._theta_p, 0)
            q_k.barrier() 
        return q_k

problem = EstimationProblem(
    state_preparation=BernoulliA(p_true),
    grover_operator=BernoulliQ(p_true),
    objective_qubits=[0],
)
print("Problem defined ✅")

# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Parameters + Helper Functions  (fixed)
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
from joblib import Parallel, delayed


# ── Ceiling N_max ──────────────────────────────────────────────────────────────
# Rounds down to prevent from overshooting the theoretical limit
def N_max_Chernoff(epsilon, alpha):
    raw = (32 / (1 - 2 * np.sin(np.pi / 14)) ** 2) \
          * np.log((2 / alpha) * np.log2(np.pi / (4 * epsilon)))
    return int(np.floor(raw))

def N_max_CP(epsilon, alpha):
    raw = 69 * np.log((2 / alpha) * np.log2(np.pi / (4 * epsilon)))
    return int(np.floor(raw))



# ══════════════════════════════════════════════════════════════════════════════
# CELL A
# ══════════════════════════════════════════════════════════════════════════════
def _safe_float(x):
    return float(x) if x is not None else None

def _safe_list(x):
    return list(x) if x is not None else None

def _safe_list_of_lists(x):
    return [list(item) for item in x] if x is not None else None

def _safe_delta_theta_list(theta_intervals):
    """
    Compute Δθ_t = θ_hi - θ_lo for every round t = 1 … T.
    Returns a list of floats (one per round), or None if unavailable.
    Malformed individual rounds are stored as None to preserve indexing.
    """
    if theta_intervals is None or len(theta_intervals) == 0:
        return None
    result = []
    for interval in theta_intervals:
        if interval is not None and len(interval) == 2:
            result.append(float(interval[1] - interval[0]))
        else:
            result.append(None)   # malformed round — keep slot
    return result


def single_diag_run_full(eps, n_shots):
    """
    One IQAE run at (eps, p_noise) with shot budget n_shots.
    Captures the full IterativeAmplitudeEstimationResult, excluding
    circuit_results (raw statevector/counts — not needed here).
    Each field is converted defensively so a single None/missing field
    doesn't discard the whole run.
    """
    try:
        iae     = IterativeAmplitudeEstimation(
            epsilon_target = eps,
            alpha          = ALPHA,
            sampler = Sampler(options={"shots": n_shots}),
            confint_method = "beta",  # chernoff or beta (CH or CP)
        )
        result = iae.estimate(problem)

        powers   = list(result.powers) if result.powers is not None else []
        K_values = [2 * k + 1 for k in powers]

        estimate = _safe_float(result.estimation_processed)
        success  = int(estimate is not None and abs(estimate - p_true) <= eps)

        # ── theta intervals — build once, derive delta_theta from it ──────────
        theta_intervals    = _safe_list_of_lists(result.theta_intervals)
        delta_theta_list   = _safe_delta_theta_list(theta_intervals)

        return {
            "success"                      : success,
            "powers"                       : powers,
            "K_values"                     : K_values,
            "T_eps"                        : len(powers),
            "max_K"                        : max(K_values) if K_values else None,
            "sum_K"                        : sum(K_values) if K_values else None,

            # amplitude-domain results
            "estimation"                   : estimate,
            "signed_error"                 : (estimate - p_true) if estimate is not None else None,
            "confidence_interval"          : _safe_list(result.confidence_interval),
            "confidence_interval_processed": _safe_list(result.confidence_interval_processed),
            "estimate_intervals"           : _safe_list_of_lists(result.estimate_intervals),

            # angle-domain results
            "theta_intervals"              : theta_intervals,
            "delta_theta_list"             : delta_theta_list,              # NEW: [Δθ_1, …, Δθ_T]

            # schedule / information mechanics
            "ratios"                       : _safe_list(result.ratios),
            "num_oracle_queries"           : int(result.num_oracle_queries) if result.num_oracle_queries is not None else None,

            # precision bookkeeping
            "epsilon_target"               : _safe_float(result.epsilon_target),
            "epsilon_estimated"            : _safe_float(result.epsilon_estimated),
            "epsilon_estimated_processed"  : _safe_float(result.epsilon_estimated_processed),
            "alpha_echo"                   : _safe_float(result.alpha),

            # sampling bookkeeping
            "shots"                        : result.shots,
            "n_shots_requested"            : n_shots,

            "error_message"                : None,
        }

    except Exception as e:
        err_str = f"{type(e).__name__}: {e}"
        print(f"[WARN] single_diag_run_full failed: {err_str}")
        return {
            "success": 0, "powers": [], "K_values": [], "T_eps": 0,
            "max_K": None, "sum_K": None,
            "estimation": None, "signed_error": None,
            "confidence_interval": None, "confidence_interval_processed": None,
            "estimate_intervals": None,
            "theta_intervals"              : None,
            "delta_theta_list"             : None,                          # NEW
            "ratios": None, "num_oracle_queries": None,
            "epsilon_target": eps, "epsilon_estimated": None,
            "epsilon_estimated_processed": None, "alpha_echo": ALPHA,
            "shots": None, "n_shots_requested": n_shots,
            "error_message": err_str,
        }

# ══════════════════════════════════════════════════════════════════════════════
# CELL B  - Simulation sweep: run R runs per (ε, p*) point, collect full results
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter
from joblib import Parallel, delayed
import numpy as np

R = 1000   # runs per point
R         = 1000
ALPHA     = 0.05
N_JOBS    = 18

epsilon_values = np.arange(0.0001, np.pi/8, 0.001).tolist()


diag_results = []

for eps in epsilon_values:
    n_shots_diag = int(N_max_CP(eps, ALPHA))   # ← Cp or CH

    print(f"\n{'─'*60}")
    print(f"ε={eps:.6f}  | N_shots={n_shots_diag} "
          f"| running {R} runs ...")

    runs = Parallel(n_jobs=N_JOBS)(
        delayed(single_diag_run_full)(eps, n_shots_diag)
        for _ in range(R)
    )

    # ── Live per-run schedule output ──────────────────────────────────────────
    for i, run in enumerate(runs):
        if run["error_message"] is not None:
            print(f"  run {i+1:>3} | ERROR: {run['error_message']}")
            continue

        status  = "✅" if run["success"] else "❌"
        err_str = f"{run['signed_error']:.6f}" if run["signed_error"] is not None else "N/A"

        # final Δθ — last element of the stored list
        dtl     = run.get("delta_theta_list")
        dth_str = f"{dtl[-1]:.6f}" if (dtl and dtl[-1] is not None) else "N/A"

        print(f"  run {i+1:>3} | {status} | T={run['T_eps']:>2} | "
              f"K_values={run['K_values']} | signed_error={err_str} | "
              f"Δθ_final={dth_str} | Δθ_seq={dtl}")

    # ── Aggregate ─────────────────────────────────────────────────────────────
    valid_runs   = [r for r in runs if r["T_eps"] > 0]
    success_rate = np.mean([r["success"] for r in runs])

    T_values    = [r["T_eps"] for r in valid_runs]
    maxK_values = [r["max_K"] for r in valid_runs]
    sumK_values = [r["sum_K"] for r in valid_runs]
    err_values  = [r["signed_error"] for r in valid_runs
                   if r["signed_error"] is not None]

    # ── delta_theta aggregation — from delta_theta_list ───────────────────────
    # final Δθ per run (last element of each run's list)
    dtheta_final_values = [
        r["delta_theta_list"][-1]
        for r in valid_runs
        if r.get("delta_theta_list") and r["delta_theta_list"][-1] is not None
    ]

    # full Δθ sequences — one list per run (for downstream convergence plots)
    dtheta_sequences = [
        r["delta_theta_list"]
        for r in valid_runs
        if r.get("delta_theta_list") is not None
    ]

    schedule_counts = Counter(tuple(r["K_values"]) for r in valid_runs)
    most_common_schedule, most_common_count = (
        schedule_counts.most_common(1)[0] if schedule_counts else ((), 0)
    )

    diag_results.append({
        "epsilon"              : eps,
        "n_shots"              : n_shots_diag,
        "success_rate"         : success_rate,
        "T_mean"               : float(np.mean(T_values))    if T_values    else None,
        "T_std"                : float(np.std(T_values))     if T_values    else None,
        "max_K_mean"           : float(np.mean(maxK_values)) if maxK_values else None,
        "sum_K_mean"           : float(np.mean(sumK_values)) if sumK_values else None,
        "mean_signed_error"    : float(np.mean(err_values))  if err_values  else None,
        # ── Δθ final (scalar summary) ─────────────────────────────────────────
        "delta_theta_mean"     : float(np.mean(dtheta_final_values))   if dtheta_final_values else None,
        "delta_theta_std"      : float(np.std(dtheta_final_values))    if dtheta_final_values else None,
        "delta_theta_median"   : float(np.median(dtheta_final_values)) if dtheta_final_values else None,
        "n_dtheta_collected"   : len(dtheta_final_values),
        # ── Δθ sequences (full per-round lists) ──────────────────────────────
        "delta_theta_sequences": dtheta_sequences,   # list of lists — one per valid run
        # ─────────────────────────────────────────────────────────────────────
        "most_common_schedule" : list(most_common_schedule),
        "most_common_count"    : most_common_count,
        "n_valid_runs"         : len(valid_runs),
        "raw_runs"             : runs,
    })

    print(f"  → success_rate={success_rate:.3f} | "
          f"Δθ_final mean={diag_results[-1]['delta_theta_mean']:.6f} ± "
          f"{diag_results[-1]['delta_theta_std']:.6f} | "
          f"most_common_schedule={list(most_common_schedule)} "
          f"({most_common_count}/{len(valid_runs)})")

print("\n✅ Full diagnostic sweep complete.")

# ══════════════════════════════════════════════════════════════════════════════
# CELL C  (updated — uses delta_theta_list from Cell A)
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np

eps_out              = []
T_out                = []
K_values_out         = []
ratios_out           = []
success_out          = []
n_shots_diag_out     = []
signed_error_out     = []   # per-run signed error        (None if unavailable)
error_message_out    = []   # per-run error message       (None if clean run)
delta_theta_out      = []   # final Δθ per run            (None if unavailable)
delta_theta_list_out = []   # full Δθ sequence per run    (None if unavailable)

# ── Flatten all runs across all epsilon points ────────────────────────────────
for point in diag_results:
    eps_val          = point["epsilon"]
    n_shots_diag_val = point["n_shots"]

    for run in point["raw_runs"]:
        eps_out.append(eps_val)
        T_out.append(run["T_eps"])
        K_values_out.append(run["K_values"])
        ratios_out.append(run["ratios"])
        success_out.append(bool(run["success"]))
        n_shots_diag_out.append(n_shots_diag_val)
        signed_error_out.append(run.get("signed_error"))       # None if errored
        error_message_out.append(run.get("error_message"))     # None if clean

        # ── Δθ — read directly from Cell A's stored list ─────────────────────
        dtl = run.get("delta_theta_list")
        delta_theta_list_out.append(dtl)                       # full sequence or None
        delta_theta_out.append(
            dtl[-1] if (dtl and dtl[-1] is not None) else None # final value  or None
        )

# ── Convert to numpy arrays ───────────────────────────────────────────────────
eps_arr              = np.array(eps_out)
T_arr                = np.array(T_out)
K_values_arr         = np.array(K_values_out,          dtype=object)  # ragged
ratios_arr           = np.array(ratios_out,             dtype=object)  # ragged
success_arr          = np.array(success_out,            dtype=bool)
n_shots_diag_arr     = np.array(n_shots_diag_out,       dtype=int)
signed_error_arr     = np.array(signed_error_out,       dtype=object)  # float or None
error_message_arr    = np.array(error_message_out,      dtype=object)  # str   or None
delta_theta_arr      = np.array(delta_theta_out,        dtype=object)  # float or None
delta_theta_list_arr = np.array(delta_theta_list_out,   dtype=object)  # list  or None

# ── Summary printout ──────────────────────────────────────────────────────────
n_total      = len(eps_arr)
n_errored    = int(np.sum([m is not None for m in error_message_out]))
n_success    = int(np.sum(success_arr))
n_fail_clean = int(np.sum(~success_arr)) - n_errored
n_dtheta     = int(np.sum([v is not None for v in delta_theta_out]))
n_dtheta_seq = int(np.sum([v is not None for v in delta_theta_list_out]))

print(f"Exported {n_total} run records across {len(diag_results)} epsilon points.")
print(f"  Failed runs (error_message set)  : {n_errored}")
print(f"  Successful runs                  : {n_success}")
print(f"  Failed (no error, just ❌)       : {n_fail_clean}")
print(f"  Runs with valid Δθ final         : {n_dtheta}     / {n_total}")
print(f"  Runs with valid Δθ sequence      : {n_dtheta_seq} / {n_total}")

# ── Save to disk ──────────────────────────────────────────────────────────────
np.savez(
    f"ceilingsweepCP_Nmax, a = {p_true}, noiseless.npz",
    epsilon          = eps_arr,
    T                = T_arr,
    K_values         = K_values_arr,
    ratios           = ratios_arr,
    success          = success_arr,
    n_shots_diag     = n_shots_diag_arr,
    signed_error     = signed_error_arr,
    error_message    = error_message_arr,
    delta_theta      = delta_theta_arr,        # scalar final Δθ per run
    delta_theta_list = delta_theta_list_arr,   # full [Δθ_1, …, Δθ_T] per run
    allow_pickle     = True,
)

print("✅ Saved — all per-run fields recorded.")


All imports OK ✅
Problem defined ✅

────────────────────────────────────────────────────────────
ε=0.000100  | N_shots=431 | running 1000 runs ...
  run   1 | ✅ | T= 6 | K_values=[1, 1, 7, 61, 657, 7101] | signed_error=0.000000 | Δθ_final=0.000003 | Δθ_seq=[0.25, 0.022505590916048673, 0.003212515524835674, 0.0003685061429521963, 3.4218916594608295e-05, 3.1716220816879748e-06]
  run   2 | ✅ | T= 6 | K_values=[1, 1, 7, 67, 703, 7639] | signed_error=0.000004 | Δθ_final=0.000003 | Δθ_seq=[0.25, 0.022510108013564373, 0.00321208935208675, 0.00033699521053509596, 3.200786550433543e-05, 2.983450291843548e-06]
  run   3 | ✅ | T= 5 | K_values=[1, 1, 7, 67, 719] | signed_error=-0.000015 | Δθ_final=0.000031 | Δθ_seq=[0.25, 0.022509321677077476, 0.00321208935208675, 0.0003371611042028433, 3.1335536194054825e-05]
  run   4 | ✅ | T= 5 | K_values=[1, 1, 9, 91, 947] | signed_error=0.000026 | Δθ_final=0.000024 | Δθ_seq=[0.25, 0.022500894026403706, 0.002504169890239577, 0.00024714252877255927, 2.37433211

KeyboardInterrupt: 